In [3]:
from langchain_core.runnables import RunnableLambda

# 1. A standard, "dumb" Python function
def clean_text(text: str) -> str:
    return text.strip().lower()

# 2. Upgrading it to a Runnable node
runnable_cleaner = RunnableLambda(clean_text)

# 3. Now it natively supports batching, async, and piping!
results = runnable_cleaner.batch(["  User1  ", "USER2   ", "  user3"])
print(results)
print(runnable_cleaner.invoke("asdfsdhf "))

['user1', 'user2', 'user3']
asdfsdhf


In [13]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnableLambda


## RunnableSequence

In [5]:
load_dotenv()
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.2)

# 1. The Nodes
prompt = PromptTemplate.from_template("Explain the concept of {topic} in one sentence.")
parser = StrOutputParser()

# 2. The Syntactic Sugar (The normal way)
chain_sugar = prompt | llm | parser

# 3. The Explicit Architecture (What happens under the hood)
chain_explicit = RunnableSequence(first=prompt, middle=[llm], last=parser)

# Both execute exactly the same way:
print(chain_explicit.invoke({"topic": "Quantum Entanglement"}))
print(chain_sugar.invoke({"topic": "Quantum Entanglement"}))

Quantum entanglement is a phenomenon where two or more particles become connected in such a way that their properties, such as spin or energy, become correlated and can be instantaneously affected by changes to the other particles, regardless of the distance between them.
Quantum entanglement is a phenomenon in which two or more particles become connected in such a way that their properties, such as spin or energy, become correlated and can be instantaneously affected by each other, regardless of the distance between them.


## RunnableParallel

In [7]:
# 1. Define distinct, specialized prompts
summary_prompt = PromptTemplate.from_template("Summarize this text in 10 words: {text}")
sentiment_prompt = PromptTemplate.from_template("Is the sentiment positive, negative, or neutral? Text: {text}")
entity_prompt = PromptTemplate.from_template("List the proper nouns in this text: {text}")

# 2. Build the individual isolated chains (Sequences)
summary_chain = summary_prompt | llm | StrOutputParser()
sentiment_chain = sentiment_prompt | llm | StrOutputParser()
entity_chain = entity_prompt | llm | StrOutputParser()

# 3. Architect the Parallel Graph
# We pass a dictionary where the keys are our chosen output names, 
# and the values are the chains we want to execute.
parallel_processor = RunnableParallel({
    "executive_summary": summary_chain,
    "emotional_tone": sentiment_chain,
    "key_entities": entity_chain
})

# 4. Execution
# A single network call spins up three parallel threads.
payload = {"text": "Google recently released Gemini 1.5, revolutionizing the AI industry with a massive context window, though some developers remain skeptical about the pricing model."}

results = parallel_processor.invoke(payload)

# 5. The output is a cleanly mapped dictionary
print(f"Summary: {results['executive_summary']}")
print(f"Tone: {results['emotional_tone']}")
print(f"Entities: {results['key_entities']}")

Summary: Google releases Gemini 1.5 with massive context window and pricing concerns.
Tone: The sentiment of the text is generally positive, but with a neutral/ slightly negative undertone. The phrase "revolutionizing the AI industry" suggests a positive impact, but the phrase "some developers remain skeptical" introduces a negative note, specifically about the pricing model. Overall, the tone is cautiously optimistic.
Entities: The proper nouns in the given text are:

1. Google
2. Gemini


## Output Parsers

In [8]:
# Executing WITHOUT a parser
raw_response = llm.invoke("What is 2+2?")

print(type(raw_response)) 
print(raw_response)

<class 'langchain_core.messages.ai.AIMessage'>
content='2 + 2 = 4.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 42, 'total_tokens': 51, 'completion_time': 0.007074022, 'completion_tokens_details': None, 'prompt_time': 0.002532597, 'prompt_tokens_details': None, 'queue_time': 0.006130566, 'total_time': 0.009606619}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019cfc5a-e9ad-72f1-8acd-1bce0412df21-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 42, 'output_tokens': 9, 'total_tokens': 51}


In [9]:
print(raw_response.content)

2 + 2 = 4.


In [11]:
parser = StrOutputParser()
print(parser.invoke(raw_response))

2 + 2 = 4.


In [15]:
prompt = PromptTemplate.from_template("Tell me a fact about {topic}")
def format_input(user_string: str) -> str:
    return user_string.strip().lower()

# -------------------------------------------------------------------
# Explicitly defining every type (No Coercion)
# -------------------------------------------------------------------
explicit_chain = (
    RunnableParallel(topic=RunnableLambda(format_input))
    | prompt 
    | llm 
    | StrOutputParser()
)

print(explicit_chain.invoke("   PYTHON PROGRAMMING   "))

Here's a fact about Python programming:

Python is named after the British comedy group Monty Python's Flying Circus. The creator of Python, Guido van Rossum, was a fan of the group and chose the name as a humorous reference.


In [16]:
# -------------------------------------------------------------------
# Leveraging LCEL Type Coercion
# -------------------------------------------------------------------
# - The dictionary {} is coerced into RunnableParallel
# - The raw function `format_input` is coerced into RunnableLambda
coerced_chain = (
    {"topic": format_input} 
    | prompt 
    | llm 
    | StrOutputParser()
)

# Execution is identical
print(coerced_chain.invoke("   QUANTUM COMPUTING   "))

One interesting fact about quantum computing is that it uses a phenomenon called superposition to process information. In classical computing, a bit can be either 0 or 1, but in quantum computing, a qubit (quantum bit) can exist in a state of superposition, meaning it can be both 0 and 1 at the same time. This allows a quantum computer to process multiple possibilities simultaneously, making it potentially much faster than a classical computer for certain types of calculations.
